# Демо: улучшение спутниковых снимков через Spark + HDFS

Этот ноутбук демонстрирует полный цикл:
1. Генерация синтетического тестового снимка.
2. Запуск распределённой обработки через PySpark.
3. Сохранение исходного и обработанного изображений в HDFS.
4. Визуальное сравнение результатов и метрики качества (PSNR/SSIM).

## 1. Подготовка тестовых данных

In [1]:
import sys, os
sys.path.insert(0, '/app')

from pathlib import Path
import subprocess

sample_path = Path('/app/data/samples/sat_demo.png')
if not sample_path.exists():
    subprocess.run(['python', '/app/scripts/generate_sample.py',
                    '--output', str(sample_path), '--size', '1536'], check=True)
#print(f'Тестовый снимок: {sample_path} ({sample_path.stat().st_size // 1024} KB)')

Тестовый снимок: /app/data/samples/sat_demo.png (91580 KB)


In [2]:
!pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 509.2 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 1.0 MB/s eta 0:00:00a 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 1.1 MB/s eta 0:00:00a 0:00:01
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [4]:
from PIL import Image
#import matplotlib.pyplot as plt

#img = Image.open(sample_path)
#plt.figure(figsize=(8, 8))
#plt.imshow(img)
#plt.title(f'Исходный снимок ({img.size[0]}x{img.size[1]})')
#plt.axis('off')
#plt.show()

## 2. Запуск пайплайна через PySpark

Под капотом: валидация → тайлинг (512×512 с overlap 32) → `mapPartitions` по Spark-воркерам → gaussian-сборка → запись в HDFS.

In [5]:
import subprocess, os
print("JAVA_HOME:", os.environ.get("JAVA_HOME", "не задан"))
print("which java:", subprocess.run(["which", "java"], capture_output=True, text=True).stdout.strip())
print("readlink:", subprocess.run(["readlink", "-f", "/usr/bin/java"], capture_output=True, text=True).stdout.strip())
print("ls /usr/lib/jvm:", subprocess.run(["ls", "-la", "/usr/lib/jvm/"], capture_output=True, text=True).stdout.strip() if os.path.exists("/usr/lib/jvm") else "директория отсутствует")

JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64
which java: /usr/bin/java
readlink: /usr/lib/jvm/java-17-openjdk-amd64/bin/java
ls /usr/lib/jvm: total 16
drwxr-xr-x 3 root root 4096 Oct 20  2023 .
drwxr-xr-x 1 root root 4096 Oct 20  2023 ..
lrwxrwxrwx 1 root root   21 Aug 24  2023 java-1.17.0-openjdk-amd64 -> java-17-openjdk-amd64
-rw-r--r-- 1 root root 1773 Aug 24  2023 .java-1.17.0-openjdk-amd64.jinfo
drwxr-xr-x 7 root root 4096 Oct 20  2023 java-17-openjdk-amd64


In [6]:
import os
import subprocess

if not os.environ.get('JAVA_HOME') or not os.path.exists(os.environ['JAVA_HOME']):
    java_path = subprocess.check_output(['which', 'java'], text=True).strip()
    java_real = subprocess.check_output(['readlink', '-f', java_path], text=True).strip()
    detected_home = java_real.replace('/bin/java', '').replace('/jre/bin/java', '')
    
    if os.path.exists(f"{detected_home}/bin/java"):
        os.environ['JAVA_HOME'] = detected_home
        print(f"🔧 AUTO: JAVA_HOME установлен в {os.environ['JAVA_HOME']}")
    else:
        raise RuntimeError(f"Не удалось определить валидный JAVA_HOME. Найден путь: {detected_home}")

if 'CLASSPATH' not in os.environ or not os.environ['CLASSPATH']:
    result = subprocess.run(
        ['/opt/hadoop/bin/hadoop', 'classpath', '--glob'],
        text=True,
        capture_output=True,
        env=os.environ
    )
    if result.returncode != 0:
        raise RuntimeError(f"hadoop classpath failed (exit code {result.returncode}):\n{result.stderr.strip()}")
    os.environ['CLASSPATH'] = result.stdout.strip()
    print(f'✅ CLASSPATH настроен ({len(os.environ["CLASSPATH"])} символов)')

🔧 AUTO: JAVA_HOME установлен в /usr/lib/jvm/java-17-openjdk-amd64
✅ CLASSPATH настроен (19474 символов)


In [7]:
# Убедимся, что CLASSPATH настроен для pyarrow.HadoopFileSystem
import os
if 'CLASSPATH' not in os.environ or not os.environ['CLASSPATH']:
    cp = subprocess.check_output(['/opt/hadoop/bin/hadoop', 'classpath', '--glob']).decode().strip()
    os.environ['CLASSPATH'] = cp
print('CLASSPATH настроен:', bool(os.environ.get('CLASSPATH')))

CLASSPATH настроен: True


In [8]:
from src.main import run_enhancement_pipeline

result = run_enhancement_pipeline(
    image_path=str(sample_path),
    image_id='SAT_DEMO_001',
    config_path='/app/config/settings.yaml',
)
result

2026-04-29 18:54:57,629 | INFO     | src.preprocessing.tiling | Валидация пройдена: sat_demo.png (10438x9991)


/opt/conda/envs/py38/lib/python3.8/site-packages/PIL/Image.py:3368: DecompressionBombWarning: Image size (104286058 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


2026-04-29 18:55:05,097 | INFO     | src.preprocessing.tiling | Изображение загружено: shape=(9991, 10438, 3) dtype=uint8
2026-04-29 18:55:05,229 | INFO     | src.preprocessing.tiling | Сформировано 462 тайлов (grid=21x22, tile=512, overlap=32)
2026-04-29 18:55:05,252 | INFO     | SatlasImageEnhancement | Подготовлено 462 тайлов для Spark
2026-04-29 18:55:08,334 | INFO     | SatlasImageEnhancement | SparkContext готов: SatlasImageEnhancement
2026-04-29 18:55:29,081 | INFO     | SatlasImageEnhancement | Память ДО инференса: 644.1 MB
2026-04-29 18:55:32,312 | INFO     | SatlasImageEnhancement | Инференс завершён: 462 тайлов за 3.23 сек (143.04 тайлов/сек)
2026-04-29 18:55:32,313 | INFO     | SatlasImageEnhancement | Память ПОСЛЕ collect: 695.8 MB
2026-04-29 18:55:33,905 | INFO     | SatlasImageEnhancement | Память ПЕРЕД сборкой: 952.3 MB
2026-04-29 18:55:55,121 | INFO     | src.postprocessing.assembly | Собрано изображение 10438x9991 из 462 тайлов (blending=gaussian)
2026-04-29 18:55:55,

/opt/conda/envs/py38/lib/python3.8/site-packages/PIL/Image.py:3368: DecompressionBombWarning: Image size (104286058 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


2026-04-29 18:56:03,340 | INFO     | SatlasImageEnhancement | Изображение большое, считаем метрики на даунскейле 2x
2026-04-29 18:56:05,693 | INFO     | SatlasImageEnhancement | Метрики: PSNR=59.40 dB, SSIM=1.0000
2026-04-29 18:56:07,107 | INFO     | src.storage.hdfs_client | Connected to HDFS: namenode:9000 via pyarrow
2026-04-29 18:56:09,458 | INFO     | src.storage.hdfs_client | Uploaded to HDFS: /app/data/samples/sat_demo.png -> /data/input/2026-04-29/SAT_DEMO_001_original.png
2026-04-29 18:56:29,291 | INFO     | src.storage.hdfs_client | Written to HDFS: /data/output/2026-04-29/SAT_DEMO_001_enhanced.png (108377917 bytes)
2026-04-29 18:56:29,722 | INFO     | src.storage.hdfs_client | Written to HDFS: /data/metadata/2026-04-29/SAT_DEMO_001.json (704 bytes)
2026-04-29 18:56:29,724 | INFO     | SatlasImageEnhancement | Результаты сохранены в HDFS: /data/metadata/2026-04-29/SAT_DEMO_001.json
2026-04-29 18:56:30,357 | INFO     | SatlasImageEnhancement | SparkSession остановлен


{'image_id': 'SAT_DEMO_001',
 'original_path': '/data/input/2026-04-29/SAT_DEMO_001_original.png',
 'enhanced_path': '/data/output/2026-04-29/SAT_DEMO_001_enhanced.png',
 'metadata_path': '/data/metadata/2026-04-29/SAT_DEMO_001.json',
 'original_shape': [9991, 10438, 3],
 'enhanced_shape': [9991, 10438, 3],
 'tile_size': 512,
 'overlap': 32,
 'num_tiles': 462,
 'inference_time_sec': 3.23,
 'total_time_sec': 91.692,
 'model': 'sentinel1_swinb_si',
 'use_mock': False,
 'metrics': {'psnr_db': 59.3975088378686, 'ssim': 0.9999873996430964},
 'processed_at_utc': '2026-04-29T18:56:05.731839+00:00',
 'error': None,
 'local_fallback': False}

## 3. Проверка содержимого HDFS

In [ ]:
# Выводим дерево /data в HDFS
out = subprocess.run(
    ['hdfs', 'dfs', '-ls', '-R', '/data'],
    capture_output=True, text=True,
    env={**os.environ, 'HADOOP_CONF_DIR': '/opt/hadoop/etc/hadoop'},
)
print(out.stdout or out.stderr)

## 4. Визуальное сравнение: исходник vs. обработанный

In [ ]:
import numpy as np
from src.storage import HDFSClient, HDFSConfig
from src.utils import load_config
import io

cfg = load_config('/app/config/settings.yaml')
hdfs = HDFSClient(HDFSConfig(
    root=cfg['storage']['hdfs_root'],
    input_dir=cfg['storage']['input_dir'],
    output_dir=cfg['storage']['output_dir'],
    metadata_dir=cfg['storage']['metadata_dir'],
))

enhanced_bytes = hdfs.get_bytes(result['enhanced_path'])
enhanced = np.asarray(Image.open(io.BytesIO(enhanced_bytes)))
original = np.asarray(Image.open(sample_path).convert('RGB'))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(original);  axes[0].set_title('Исходник');    axes[0].axis('off')
axes[1].imshow(enhanced);  axes[1].set_title('Обработанный'); axes[1].axis('off')
# Кроп для детального сравнения
crop = slice(400, 700)
axes[2].imshow(np.hstack([original[crop, crop], enhanced[crop, crop]]))
axes[2].set_title('Кроп 300x300: orig | enhanced'); axes[2].axis('off')
plt.tight_layout(); plt.show()

print(f"PSNR = {result['metrics']['psnr_db']:.2f} dB")
print(f"SSIM = {result['metrics']['ssim']:.4f}")
print(f"Время инференса:   {result['inference_time_sec']} сек")
print(f"Полное время:      {result['total_time_sec']} сек")
print(f"Обработано тайлов: {result['num_tiles']}")